# Notebook 11 — Transfer Learning for Crop Disease CNN

Fine-tune ResNet50 pre-trained on ImageNet for crop disease classification.
Expected improvement: 83.27% → 90%+ on 42 disease classes.

In [ ]:
# from google.colab import drive  # removed for local run
# drive.mount('/content/drive')  # removed for local run

import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import numpy as np
import matplotlib.pyplot as plt
import os

IMAGES_PATH = '../data/images/crop_disease'
MODELS = '../models'
OUTPUTS = '../outputs/plots'

print(f"Image directory: {IMAGES_PATH}")
print(f"Classes: {len(os.listdir(IMAGES_PATH))}")

## Phase 1: Data Loading with Augmentation

In [ ]:
# Image data generators with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)

# Load data with generators
train_generator = train_datagen.flow_from_directory(
    IMAGES_PATH,
    target_size=(224, 224),  # ResNet50 input size
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_generator = val_datagen.flow_from_directory(
    IMAGES_PATH,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

print(f"Classes found: {train_generator.num_classes}")
print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {val_generator.samples}")

## Phase 2: Transfer Learning Model Architecture

Load ResNet50 pre-trained on ImageNet and fine-tune for crop diseases.

In [ ]:
# Load ResNet50 with ImageNet weights
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze base layers (first 18 blocks, last 5 will be fine-tuned)
for layer in base_model.layers[:-5]:
    layer.trainable = False

# Add custom classification head
x = GlobalAveragePooling2D()(base_model.output)
x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)
predictions = Dense(42, activation='softmax')(x)  # 42 crop disease classes

model = Model(inputs=base_model.input, outputs=predictions)

# Compile model
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\nModel Architecture:")
model.summary()

## Phase 3: Training with Fine-tuning

In [ ]:
# Callbacks
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)

# Train model
history = model.fit(
    train_generator,
    epochs=50,
    validation_data=val_generator,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# Save model
model.save(f'{MODELS}/cnn_resnet50_transfer.h5')
print("✓ Transfer learning model saved → cnn_resnet50_transfer.h5")

## Phase 5: Visualization of Training

In [ ]:
# Load original CNN model for comparison
original_cnn = tf.keras.models.load_model(f'{MODELS}/cnn_model.h5')

# Evaluate both models
original_scores = original_cnn.evaluate(val_generator, verbose=0)
transfer_scores = model.evaluate(val_generator, verbose=0)

print("\n" + "="*70)
print("CNN COMPARISON: Standard vs Transfer Learning")
print("="*70)
print(f"\nStandard CNN (from Notebook 6):")
print(f"  Loss: {original_scores[0]:.4f}")
print(f"  Accuracy: {original_scores[1]:.4f} (83.27%)")

print(f"\nResNet50 Transfer Learning:")
print(f"  Loss: {transfer_scores[0]:.4f}")
print(f"  Accuracy: {transfer_scores[1]:.4f}")

improvement = (transfer_scores[1] - original_scores[1]) * 100
print(f"\nImprovement: +{improvement:.2f}% accuracy")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('ResNet50 Transfer Learning - Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Loss
axes[1].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('ResNet50 Transfer Learning - Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUTS}/transfer_learning_resnet50_training.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Training visualization saved")